# Baltic States Synchronisation and the LT&ndash;PL Link, 2025

**2025-02-07/09** &ndash; Estonia, Latvia, and Lithuania desynchronised from Russia's BRELL ring and synchronised with Continental Europe. ([ENTSO-E announcement](https://www.entsoe.eu/news/2025/02/09/entso-e-confirms-successful-synchronization-of-the-continental-european-electricity-system-with-the-systems-of-the-baltic-countries/))

We'll have a look at how cross-border flows changed throughout the process.

**Before you run this analysis:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from common import (
    ast_price_convergence_chart,
    before_after_convergence,
    before_after_convergence_chart,
    convergence_summary,
    fetch_flows,
    flow_duration_curve,
    flow_price_scatter,
    flow_type_share_pies,
    is_baltic_border,
    litgrid_capacity_plan_chart,
    save_chart,
    voltpath_convergence_chart,
)

In [ ]:
# Key constants
BALTIC_EVENT = {
    "name": "Baltic sync",
    "slug": "baltic-sync",
    "date": "2025-02-08",
    "date_from": "2025-02-04",
    "date_to": "2025-02-12",
}

## Did the sync show up in the data?

The Baltics have run on Russia's BRELL ring since Soviet times; desynchronising means Continental Europe's frequency, not Moscow's, now governs their grids &ndash; a physical grid change, not a market-rule change like the coupling-algorithm updates the rest of this series covers. That distinction matters for what to expect in the data: a real topology change should show up in physical constraints (congestion, capacity ceilings) as much as in market outcomes (price convergence).

The window below is used to cover the synchronisation's date &plusmn;4 days, fetched at an hourly resolution.

In [ ]:
baltic_event = BALTIC_EVENT
baltic_df = fetch_flows(baltic_event["date_from"], baltic_event["date_to"], resolution="1H")

### Capacity ceiling or loop flow?

TSOs typically allocate less transfer capacity right after a cutover as temporary caution. As a result, congestion goes up and convergence goes down which, however, can also consistent with a second story: physical loop flows on the newly-meshed AC grid overriding the commercial schedule – a real inefficiency, not just caution.
 
To help to distinguish between these scenarios, Voltpath classifies flow into these categories from the flow/price relationship directly:

- congestion is flow pinned at the interconnector's NTC limit
- counter_intuitive (CI) is flow running against the price signal

CI flows are commonly caused by loop flow ([Voltpath docs: flow types](https://docs.voltpath.ai/concepts/flow-types)), and so a [flow-duration curve](https://voltpath.ai/reference#flow-duration-curve) tells the first story apart from the second.

If the flat top (the transfer-capacity ceiling) drops after the sync, that's capacity caution. If points scatter off the price-tracking curve after the sync, flow is moving against the price signal, which is the loop-flow story.

Both charts are restricted to Baltic borders only (`EE`, `LV`, `LT` on either end) and reuse the window already fetched above.

[**How to read the curve**](https://voltpath.ai/reference#flow-duration-curve): each border's hourly flow, sorted descending, against percent of hours at or above it. A flat top before the slope starts is the border running at its transfer-capacity ceiling; compare that flat top's height before vs. after the sync.

In [ ]:
baltic_borders_df = baltic_df[baltic_df["border"].map(is_baltic_border)].copy()
baltic_borders_df["period"] = (
    baltic_borders_df["datetime"].lt(baltic_event["date"]).map({True: "before", False: "after"})
)

fig = flow_duration_curve(
    baltic_borders_df,
    f"Baltic borders: flow duration curve, 4 days before vs. after {baltic_event['date']}",
)
save_chart(fig, f"baltic-sync-flow-duration-curve-{baltic_event['date_from']}-to-{baltic_event['date_to']}")
fig.show()

[**How to read the scatter charts**](https://voltpath.ai/reference#flow-price-scatter): each point is one hourly flow observation, price delta on the x-axis, net flow on the y-axis. Points following the diagonal are tracking the price signal (Convergence/Congestion); points sitting in the shaded Counter-intuitive quadrants are moving against it &ndash; the loop-flow signature.

Very few points fall in the Counter-intuitive quadrants, confirming [what happened in fact](https://ceenergynews.com/electricity/litgrid-electricity-capacity-poland/): TSOs allocated less transfer capacity to keep grid operations safe, rather than flow running against the price signal.

In [ ]:
fig = flow_price_scatter(
    baltic_borders_df,
    f"Baltic borders: flow vs. price delta, before vs. after {baltic_event['date']}",
)
save_chart(fig, f"baltic-sync-flow-price-scatter-{baltic_event['date_from']}-to-{baltic_event['date_to']}")
fig.show()

### Did market-coupling reliability actually change?

Examining the share of hours with convergent flow change before and after the sync date splits the same time-share metric ENTSO-E itself uses to attribute past market-coupling effects (e.g. [Core FBMC's 2022 impact on Croatian/Austrian congestion income](https://eepublicdownloads.blob.core.windows.net/public-cdn-container/clean-documents/Publications/Position%20papers%20and%20reports/2025/ENTSO-E_Bidding_Zone_Configuration_Technical_Report_2025.pdf
) - pp. 62-83).

In [ ]:
comparison = before_after_convergence(baltic_df, baltic_event["date"])
fig = before_after_convergence_chart(comparison, baltic_event)
save_chart(
    fig,
    f"{baltic_event['slug']}-convergence-before-after-{baltic_event['date_from']}-to-{baltic_event['date_to']}",
)
fig.show()

Same before/after split, visually: each pie is one period's full flow-type mix (all borders, not just Baltic ones - same continent-wide window as the print above), with Convergence pulled out to match the number just printed.

In [ ]:
fig = flow_type_share_pies(
    baltic_df,
    baltic_event["date"],
    f"{baltic_event['name']}: flow-type share, before vs. after",
)
save_chart(fig, f"baltic-sync-flow-type-pies-{baltic_event['date_from']}-to-{baltic_event['date_to']}")
fig.show()

## Why LT&ndash;PL specifically, and did it recover?

### What about the other two borders that shift a lot?

LT&ndash;PL is the border this section is actually about &ndash; it's the only one where sync mechanically changes the interconnection itself (async back-to-back HVDC &rarr; normal synchronous AC). EE&ndash;FI and LT&ndash;SE_4 also show a visibly different ceiling before vs. after in the chart above, but for unrelated reasons, not sync:

- **EE&ndash;FI (Estlink)** was still recovering from the Estlink 2 subsea cable sabotage of 2024-12-25 &ndash; about six weeks before this window, and a real physical outage/repair constraint, not a sync effect. ([Fingrid: EstLink 2 electricity transmission link has failed, investigation continues](https://www.fingrid.fi/en/news/news/2024/estlink-2-electricity-transmission-link-between-finland-and-estonia-has-failed-investigation-continues/))
- **LT&ndash;SE_4 (NordBalt)** is HVDC on both sides of this event and stays that way: Sweden sits in the Nordic synchronous area, which is distinct from both BRELL and CESA, so Baltic sync doesn't change how this link is electrically wired the way it did for LT&ndash;PL. The shift visible here more plausibly reflects the heightened security posture after a suspected sabotage attempt on this exact link, reported 2025-01-12, than any capacity-mechanism change. ([LRT: Lithuania's Navy steps up NordBalt cable protection amid reports of damage attempts](https://www.lrt.lt/en/news-in-english/19/2459548/lithuania-s-navy-steps-up-nordbalt-cable-protection-amid-reports-of-damage-attempts))


### AST's LV&ndash;EE price convergence, checked against our own data

AST (the Latvian TSO) reports the share of trading intervals where Latvia and Estonia had identical day-ahead prices fell 21 percentage points in 2025 vs. 2024 &ndash; but attributes this to Estonian line maintenance, not Baltic sync. Plotting it alongside the other three Baltic-region border pairs shows why that attribution is plausible: LV&ndash;LT barely moved over the same period, so this isn't a Baltic-wide effect. AST's own methodology notes confirm this is built from ENTSO-E Transparency Platform data &ndash; the same ultimate source Voltpath draws from &ndash; so a same-direction check against our own numbers is meaningful.

Source: [AST, Latvian electricity market overview, 2025 all year](https://www.ast.lv/en/electricity-market-review?year=2025&month=13).

In [ ]:
fig = ast_price_convergence_chart()
save_chart(fig, "baltic-ast-price-convergence-2024-vs-2025")
fig.show()

The chart above measures identical day-ahead price; the one below measures our own `share_converged` (flow direction relative to price) for the same four pairs and years &ndash; a different metric, so the percentages won't match, but every pair moves the same direction: down, and by a lot for FI&ndash;EE, EE&ndash;LV and LT&ndash;SE4, barely at all for LV&ndash;LT.

In [ ]:
BALTIC_PRICE_PAIRS = {
    "FI & EE": ("FI", "EE"),
    "EE & LV": ("EE", "LV"),
    "LV & LT": ("LV", "LT"),
    "LT & SE4": ("LT", "SE_4"),
}

rows = []
for label, (zone_from, zone_to) in BALTIC_PRICE_PAIRS.items():
    for year in ("2024", "2025"):
        year_df = fetch_flows(f"{year}-01-01", f"{year}-12-31", resolution="1D", zone=[zone_from, zone_to])
        pair_borders = [f"{zone_from} -> {zone_to}", f"{zone_to} -> {zone_from}"]
        summary = convergence_summary(year_df[year_df["border"].isin(pair_borders)])
        rows.append({"pair": label, "year": year, "share_pct": summary["share_converged"].mean() * 100})

voltpath_convergence = pd.DataFrame(rows)
fig = voltpath_convergence_chart(voltpath_convergence)
save_chart(fig, "baltic-voltpath-convergence-2024-vs-2025")
fig.show()

### Why LT&ndash;PL specifically, and why it recovers

The flow-duration-curve collapse traces to one border, LT&ndash;PL (LitPol Link), not a general Baltic effect. Litgrid's own switchyard rebuild moved the Lithuanian side from an async back-to-back HVDC converter to a normal synchronous AC connection at sync &ndash; so the link stopped needing DC conversion, but its commercial capacity is now bounded by a different constraint: available fast-frequency-response reserve (mostly grid-scale batteries), not the line's 500 MW thermal rating. The Baltic CCR TSOs' own capacity-calculation methodology document states this explicitly: at sync time the Baltic system had "about 200 MW" of that reserve installed, which is why the post-sync flow ceiling sits near 0&ndash;200 MW rather than tapering gradually ([AST: Baltic DA/ID CCM explanatory document, 2025-02-06, Section 7](https://ast.lv/sites/default/files/editor/Baltic_DA_ID_CCM_explanatory_document_06022025.pdf)). It's a deliberate, standing TSO capacity cap for frequency stability, not an outage or a fault.

Litgrid's own later disclosure gives the actual numbers behind that recovery: allocated capacity averaged ~170 MW export / ~150 MW import since sync (vs. ~350/492 MW pre-sync), with a staged plan to reach 365/200 MW by end-2026 and 500/353 MW from 2027 as more battery storage comes online ([Litgrid, via Reuters/MarketScreener](https://www.marketscreener.com/news/litgrid-to-increase-electricity-trading-capacity-with-poland-ce7d50d2df8ef323)).


### Litgrid's own before/after numbers, plotted, checked against our own data

The mechanism writeup above quotes Litgrid's autumn-2025 press statement, including a recovery-plan target of 365 MW export / 200 MW import by end of 2026. This turns those numbers into a chart rather than leaving them as prose, and checks the 2026 target against our own LT&ndash;PL flow data through 2026-09-24 rather than taking the plan on faith.

In [ ]:
baltic_long = fetch_flows("2024-11-01", "2026-09-24", resolution="1D", zone=["EE", "LV", "LT"])
lt_pl_2026 = baltic_long[(baltic_long["border"] == "LT -> PL") & (baltic_long["datetime"] >= "2026-01-01")]
avg_export_2026 = lt_pl_2026.loc[lt_pl_2026["net_flow_mw"] > 0, "net_flow_mw"].mean()
avg_import_2026 = lt_pl_2026.loc[lt_pl_2026["net_flow_mw"] < 0, "net_flow_mw"].abs().mean()
print(f"2026 YTD avg LT-PL export: {avg_export_2026:.0f} MW (plan target: 365 MW)")
print(f"2026 YTD avg LT-PL import: {avg_import_2026:.0f} MW (plan target: 200 MW)")

fig = litgrid_capacity_plan_chart(observed_2026=(avg_export_2026, avg_import_2026))
save_chart(fig, "baltic-litgrid-lt-pl-capacity-plan")
fig.show()

## Takeaways

- The same functions used to reproduce a year-old published analysis (see [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb)) turn a historical curiosity into a live monitoring tool: point them at a grid synchronisation event, and the before/after shift shows up the same way.
- Baltic sync is a case where congestion and convergence move together &ndash; a real physical-topology change, not a market-algorithm artifact &ndash; and the dip traces to one border, LT&ndash;PL, recovering on a schedule Litgrid has itself disclosed and that our own data now partially confirms (2026 running behind its own end-2026 target).
- This is a monitoring pattern, not a one-off: the same calls work for any future structural change, without modification.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).
